In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ee
ee.Authenticate()
ee.Initialize()

In [2]:
lat_min, lat_max = 28.0, 29.39
lat_avg = (lat_max + lat_min) / 2
lon_min, lon_max = 76.3, 79.0
lon_avg = (lon_max + lon_min) / 2

stepp = 0.25


roi = ee.Geometry.Rectangle([lon_min, lat_min, lon_max, lat_max])
pixel = ee.Geometry.Point([77.0, 29.0])
micropixel = ee.Geometry.Rectangle([lon_avg-stepp, lat_avg-stepp, lon_avg+stepp, lat_avg+stepp])
# scale = 5000
start_date = '2024-02-05'
end_date = '2024-02-06'

In [3]:
"""
Can define multiple collections (wind, SO2, NO2). Prepare what we're doing for each. Then feed outputs for all. No merging required as such, except right before training/input into a model.
"""

collection = (
    ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_NO2")
    .filterBounds(micropixel)# REPLACE WITH ROI
    .filterDate(start_date, end_date)
    .select('tropospheric_NO2_column_number_density')
)

In [4]:
def make_coarse(image):
    return (image
        .reduceResolution(
            reducer=ee.Reducer.mean(),
            bestEffort=True
        )
        .reproject(
            scale=1100,
            crs='EPSG:4326'
        )#.focal_mean(radius=2, units='pixels')
        .copyProperties(image, ['system:time_start'])
    )

coarse_collection = collection.map(make_coarse) # This (possibly coarse) collection will be henceforth be used.

size_of_coll = coarse_collection.size().getInfo()
print(f"Size of collection is {size_of_coll}. In particular, coarse_collection.getInfo()['features'] has that many items.")
print()
print(f"This means we have ~{size_of_coll} entries in the given time range. For each of these, we have an image (11px by 11px) of the entire region we have (after reducing resolution) So if we have {size_of_coll} measurements for the region, and the region is (say) 11*11 pixels, then we have ~{11*11*size_of_coll} measurements. Many of these may be null!")
# print(coarse_collection.first().getInfo()['properties']['system:time_start']) # need to parse as date-time

Size of collection is 14. In particular, coarse_collection.getInfo()['features'] has that many items.

This means we have ~14 entries in the given time range. For each of these, we have an image (11px by 11px) of the entire region we have (after reducing resolution) So if we have 14 measurements for the region, and the region is (say) 11*11 pixels, then we have ~1694 measurements. Many of these may be null!
